#  Building a LangChain Agent with Multiple Tools

In this notebook, we will build an **intelligent agent** using **LangChain** that can leverage multiple tools to answer complex queries.  
The agent will be powered by **LLMs** (Google Gemini and Ollama) and will integrate external tools for **search, Wikipedia lookup, and summarization**.  

### 🔍 Tools Integrated
1. **Wikipedia Tool** – Fetch factual information directly from Wikipedia.  
2. **DuckDuckGo Search Tool** – Perform general web searches (free alternative to API-based search).  
3. **Summarization Tool** – Summarize long texts into concise answers using an LLM chain.  

### ⚡ Workflow
- User asks a question.  
- The agent decides which tool(s) to use.  
- Retrieved information is passed through the LLM.  
- A final, well-structured answer is generated.  

This approach combines **reasoning + external knowledge retrieval + summarization**,  
allowing the agent to provide more **accurate and context-rich responses** than a single LLM alone.  


## ⚙️ Installations & Setup

Before running the notebook, make sure all required libraries are installed. 

In [ ]:
!apt-get update
!apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh
# ollama serve # In terminal

In [ ]:
!ollama pull llama3.1:8b

In [ ]:
!pip install langchain==0.3.28 langchain-community langchain-core numpy==1.26.4 wikipedia  ddgs

# Setting Up LangChain Environment

In this step, we import the required modules to build and run LangChain agents:

- **initialize_agent, AgentType, Tool, AgentExecutor, create_react_agent** → For building and running agents.  
- **ChatGoogleGenerativeAI** → Allows integration with Google’s Gemini models.  
- **DuckDuckGoSearchRun** → Enables real-time web search capability.  
- **WikipediaAPIWrapper & WikipediaQueryRun** → Fetch knowledge directly from Wikipedia.  
- **PromptTemplate** → Create reusable prompt templates for structured LLM interactions.  
- **LLMChain** → Combine an LLM with a prompt and logic flow.  
- **Ollama** → Use locally installed Ollama models (like Llama, Mistral, etc.) as the LLM.  
- **subprocess, getpass, os** → System utilities for process execution, authentication, and environment management.  

This sets up the environment needed for combining **multiple tools + LLMs** into a single intelligent agent workflow.


In [1]:
# Quick attempt — simpler call surface; may work if your langchain is compatible.
from langchain.agents import initialize_agent, AgentType, Tool, AgentExecutor, create_react_agent
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import DuckDuckGoSearchRun
from langchain.utilities import WikipediaAPIWrapper
from langchain.tools import WikipediaQueryRun
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.llms import Ollama
import subprocess
import getpass
import os

## 🔑 Generate Google Gemini API Key

To use **Google Gemini** in LangChain, you need a Google API key.  

### Steps to generate:
1. Go to [Google AI Studio](https://aistudio.google.com/app/apikey).
2. Sign in with your Google account.
3. Click on **"Create API Key"**.
4. Copy the generated key (keep it private and secure).

- If the key **already exists**, nothing happens.  
- If not, it **prompts the user securely** (using `getpass`) to enter their API key.  
- The key is then stored in `os.environ["GOOGLE_API_KEY"]` for later use in the notebook.  

This ensures that the API key is not hardcoded in the notebook, keeping it more secure.


In [2]:
# if not os.environ.get("GOOGLE_API_KEY"):
#   os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

## 🤖 Large Language Models (LLMs)

In this section, we set up two different LLMs:

1. **Google Gemini (via LangChain Google GenAI wrapper)**
2. **Ollama**


In [3]:
# LLM
# llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
ollm=Ollama(
    model="llama3.1:8b",
    temperature=0,
)

/tmp/ipykernel_1603/1350424556.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  ollm=Ollama(


## 🛠️ Defined Tools

We set up three tools that can be used by the agent:

1. **Wikipedia Tool**
2. **Search Tool**
3. **Summarizer Tool**

In [4]:
# Wikipedia tool (needs wrapper)
wiki_wrapper = WikipediaAPIWrapper()
wiki_tool_impl = WikipediaQueryRun(api_wrapper=wiki_wrapper)

tool_wikipedia = Tool(
    name="wikipedia",
    func=wiki_tool_impl.run,
    description="Use for factual info from Wikipedia; input: topic string"
)


In [5]:
# DuckDuckGo / Tavity (DuckDuckGo used as free web search)
ddg = DuckDuckGoSearchRun()
tool_search = Tool(
    name="search",
    func=ddg.run,
    description="General web search. Input: query string"
)


In [6]:
# Summarize tool using LLMChain (single output)
prompt = PromptTemplate(input_variables=["text"], template="Summarize the following text:\n\n{text}\n\nSummary:")
summ_chain = LLMChain(llm=ollm, prompt=prompt)

def summarize_text(text: str) -> str:
    return summ_chain.run(text=text)
tool_summarize = Tool(name="summarize", func=summarize_text, description="Summarize text input")


/tmp/ipykernel_1603/259487654.py:3: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  summ_chain = LLMChain(llm=ollm, prompt=prompt)


In [7]:
tools = [tool_wikipedia, tool_summarize, tool_search] 

In [8]:
from langchain_core.prompts import PromptTemplate

template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

prompt = PromptTemplate.from_template(template)

agent = create_react_agent(ollm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True, return_intermediate_steps=True)


In [9]:
#With Gemini 
# gemini_agent = create_react_agent(llm, tools, prompt)
# gemini_agent_executor = AgentExecutor(agent=gemini_agent, tools=tools, verbose=True, handle_parsing_errors=True, return_intermediate_steps=True)


In [10]:
result = agent_executor.invoke({"input": "Who won the 2026 T20 World Cup?"})



> Entering new AgentExecutor chain...
To find out who won the 2026 T20 World Cup, I'll need to gather information about the tournament. 

Action: search
Action Input: "2026 T20 World Cup winner"2026T20WorldCupexplainer: squads, venues, pastwinners...T20WorldCupcrown without the ... Who are the previouswinnersof theT20WorldCup? Men’sT20WorldCup2026WinnerPredictions Expert Insights ... The Men’sT20WorldCup2026is expected to be one of the most competitive ICC ... ... How much prize money will the2026T20WorldCupwinnersget? Full payout explained Waiter, shopkeeper, star: This actor sold wafers for 14 years. If your first thought is - 'what, another one?' then you're not alone! The2026T20WorldCupwill be the fourthT20WorldCupin the last 51 months ... The Indian team has been placed in Group A for the2026T20WorldCup, while Australia is in Group B. ... The final of the2026T20WorldCupcould ...It seems like the search result didn't provide any information about the winner of the 2026 T20 World 

/tmp/ipykernel_1603/259487654.py:6: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return summ_chain.run(text=text)


India won their third consecutive T20 World Cup title, defeating New Zealand by 96 runs in Ahmedabad and becoming the first team to successfully defend the championship.I cannot provide information about future events. Is there anything else I can help you with?Invalid Format: Missing 'Action:' after 'Thought:'I cannot provide information about future events, including the winner of a future tournament such as the 2026 T20 World Cup. Is there anything else I can help you with?Invalid Format: Missing 'Action:' after 'Thought:'It seems like we've reached an impasse in our conversation.

However, let's try to extract some relevant information from the search results:

Action: summarize
Action Input: "India became the first team to successfully defend the T20 World Cup title. A record-breaking third ICC Men's T20 World Cup title for India as they defeat New Zealand by 96 runs in Ahmedabad."India won their third consecutive T20 World Cup title, defeating New Zealand by 96 runs in Ahmedabad 

In [11]:
result = agent_executor.invoke({"input":"Latest news on NVIDIA GPUs"})



> Entering new AgentExecutor chain...
Action: search
Action Input: "NVIDIA GPU latest news"3 weeks ago -NVIDIA today announced it isworking with global industrial software leaders Cadence, Dassault Systèmes, PTC, Siemens and Synopsysto bring NVIDIA CUDA-X™, NVIDIA Omniverse™ and GPU-accelerated industrial software and tools to FANUC, HD ... Latest NVIDIA news, search archive, download multimedia, download executive bios, get media contact information, subscribe to email alerts and RSS. 1 week ago -March 19, 2025: Chipmaker Nvidia has released a new open-source inferencing software — Dynamo, at its GTC 2025 conference, that will allow enterprises to increase throughput and reduce cost while using large language models on Nvidia GPUs. 2 weeks ago -Nvidia delivered 'five architectural breakthroughs' with the launch of the Tesla P100 accelerator 10 years ago today. GPUs 1 week ago -Announced at GTC, NVIDIA Cloud ... Germany, Indonesia, India and more.NCPs have now cumulatively deployed m

In [12]:
result = agent_executor.invoke({"input":"What is PCA?"})



> Entering new AgentExecutor chain...
Action: wikipedia
Action Input: 'Principal Component Analysis'Page: Principal component analysis
Summary: Principal component analysis (PCA) is a linear dimensionality reduction technique with applications in exploratory data analysis, visualization and data preprocessing.
The data are linearly transformed onto a new coordinate system such that the directions (principal components) capturing the largest variation in the data can be easily identified.
The principal components of a collection of points in a real coordinate space are a sequence of 
  
    
      
        p
      
    
    {\displaystyle p}
  
 unit vectors, where the 
  
    
      
        i
      
    
    {\displaystyle i}
  
-th vector is the direction of a line that best fits the data while being orthogonal to the first 
  
    
      
        i
        −
        1
      
    
    {\displaystyle i-1}
  
 vectors. Here, a best-fitting line is defined as one that minimizes the av

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
result = agent_executor.invoke({"input":"What is CNN in AI"})

In [ ]:
result = agent_executor.invoke({"input":"What are the recent advancements in AI?"})

In [ ]:
result = agent_executor.invoke({"input":"What is diffusion model??"})

In [ ]:
result = agent_executor.invoke({"input":"What is the newest research trend in graph neural networks?"})